flow = '01-simple.json'
!wget "https://raw.githubusercontent.com/mldong/jeeflow-python/refs/heads/master/flows/{flow}" -O  "./flows/{flow}"

In [1]:
!pip show jeeflow 

Name: jeeflow
Version: 1.8.28
Summary: Lightweight async workflow engine - Python implementation
Home-page: 
Author: mldong
Author-email: 
License: Apache-2.0
Location: /home/gem/.local/lib/python3.12/site-packages
Requires: 
Required-by: 


In [8]:
# 0 RESET DATA ALL
!curl -X POST http://localhost:8101/jeeflow/api/reset \
  -H "Content-Type: application/json" \
  -d '{}'

{"code":0,"msg":"成功","data":null}

In [3]:
# 1. 发起（自动完成申请节点）
processInstance = !curl -s -X POST http://localhost:8101/jeeflow/wf/processDefine/startAndExecute \
  -H "Content-Type: application/json" \
  -d '{"processDefineId":1,"operator":"user1"}' | jq -r '.data.processInstanceId'
PROCESSINSTANCE = processInstance[0]
print(PROCESSINSTANCE)

91227083011468


In [4]:
# 2. 组长查待办
taskid = !curl -s -X POST http://localhost:8101/jeeflow/wf/processTask/todoList \
  -H "Content-Type: application/json" -d '{"operator":"leader"}'  | jq -r '.data.rows[0].id' # 门面契约统一 operator
TASKID = taskid[0]
print(TASKID)

91227082838319


In [5]:
# 3. 组长同意
cmd = f"""curl -s -X POST http://localhost:8101/jeeflow/wf/processTask/execute \
-H "Content-Type: application/json" \
-d '{{"processTaskId":"{TASKID}","operator":"leader","submitType":1}}'"""

# print(cmd)
!{cmd}

{"code":0,"msg":"成功","data":null}

In [6]:
# 4. 查看实例状态
cmd = f"""curl -s -X POST http://localhost:8101/jeeflow/wf/processInstance/detail \
-H "Content-Type: application/json" \
-d '{{"id":"{PROCESSINSTANCE}"}}'"""

#print(cmd)
!{cmd} | jq


{
  "code": 0,
  "msg": "成功",
  "data": {
    "id": "91227083011468",
    "parentId": null,
    "processDefineId": "1",
    "state": 10,
    "parentNodeName": "",
    "businessNo": "",
    "operator": "user1",
    "variables": {
      "u_userId": "user1",
      "u_realName": "张三",
      "u_deptId": "D01",
      "u_deptName": "研发部",
      "u_postId": "P01",
      "u_postName": "工程师",
      "autoGenTitle": "张三的简单审批流程-2026-09-11 09:09",
      "submitType": 0
    },
    "formData": {},
    "createTime": "2026-09-11T09:09:08.253624",
    "createUser": "user1",
    "jsonObject": {
      "name": "simple",
      "displayName": "简单审批流程",
      "type": "approval",
      "instanceUrl": "/form/apply",
      "nodes": [
        {
          "id": "start",
          "type": "snaker:start",
          "x": 100,
          "y": 200,
          "properties": {
            "width": 50,
            "height": 50
          },
          "text": {
            "value": "开始"
          }
        },
        {
       